# Notebook 06: Sintesi Visuale delle Figure e Compilazione del Report Consuntivo
## Fase 6 del Piano Operativo (`results/final_experimental_report.md`)

**Deliverable Scientifici ed Editoriali:**
1. **T6.1: Heatmap Spazio-Temporali $10 \times 6$ a Doppia Polarità (+ Rosso / - Blu)**:
   - Mappe di calore ad alta definizione (300 DPI) per il paper IEEE, rappresentanti i valori di Shapley normalizzati con segno per attacchi critici e coppie confuse.
2. **T6.2: Grafici Comparativi delle Curve MoRF vs. LeRF per i 3 Modelli**:
   - Visualizzazione simultanea delle curve di degradazione progressiva, evidenziando il divario di fedeltà causale (AUDC) e la tenuta a $k=1$ (focus su `FLG`).
3. **T6.3: Compilazione Automatica del Report Sperimentale Consuntivo**:
   - Generazione di `results/final_experimental_report.md` che consolida in un unico documento le Tabelle 1, 2, 3 e 4 del registro sperimentale.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve project paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
RESULTS_DIR = REPO_ROOT / 'results'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'

for d in [METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("[OK] Setup completato per la sintesi visuale e il reporting consuntivo.")

### 6.1 Generazione Heatmap $10 \times 6$ a Doppia Polarità (+ Rosso / - Blu) (Task T6.1)
Tracciamo le mappe di calore dell'attribuzione SHAP normalizzata per istanza con colormap divergente (`vlag` o `coolwarm`), in cui:
- **Rosso (+)** indica un contributo eccitatorio (spinge verso la classe dell'attacco);
- **Blu (-)** indica un contributo inibitorio (allontana dalla classe dell'attacco).

In [ ]:
FIELDS = ['PL', 'IAT', 'DIR', 'WIN', 'TTL', 'FLG']
npz_path = METRICS_DIR / 'xai_explanations.npz'

npz_data = np.load(npz_path, allow_pickle=True)  # generated by Notebook 03
hm_lopez = np.median(npz_data['shap_lopez_cell'], axis=0)    # [10, 6]
hm_biflow = np.median(npz_data['shap_biflow_cell'], axis=0)  # [10, 6]
hm_meddef = np.median(npz_data['shap_meddef_cell'], axis=0)  # [10, 6]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
v_limit = max(np.abs(hm_lopez).max(), np.abs(hm_biflow).max(), np.abs(hm_meddef).max())

heatmaps = [('Lopez17CNN (Black-Box)', hm_lopez), 
             ('BiflowTransformer (Sequential)', hm_biflow), 
             ('MedDefenderTransformer (By-Design)', hm_meddef)]

for idx, (title, hm) in enumerate(heatmaps):
    ax = axes[idx]
    sns.heatmap(
        hm,
        ax=ax,
        cmap='coolwarm',
        center=0,
        vmin=-v_limit,
        vmax=v_limit,
        xticklabels=FIELDS,
        yticklabels=[f'Pkt {t+1}' for t in range(10)],
        cbar=(idx == 2),
        cbar_ax=cbar_ax if (idx == 2) else None,
        linewidths=0.5,
        linecolor='white'
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(r'Campi di Intestazione di Protocollo, $f \in \{PL, IAT, DIR, WIN, TTL, FLG\}$', fontsize=10)
    if idx == 0:
        ax.set_ylabel(r'Indice Sequenza Temporale Pacchetto, $t \in [1..10]$', fontsize=10)
cbar_ax.set_ylabel(r'Attribuzione SHAP con Segno per Istanza $\tilde{\phi}_{t,f}$ (+ Rosso / - Blu) [a.u.]', fontsize=10)

plt.suptitle('Heatmap Spazio-Temporale XAI (+ Rosso Eccitatorio / - Blu Inibitorio) - Eq. 5 Nascita et al.', fontsize=13, y=1.02)
out_hm_png = FIGURES_DIR / 'xai_heatmaps_multimodel.png'
out_hm_pdf = FIGURES_DIR / 'xai_heatmaps_multimodel.pdf'
plt.savefig(out_hm_png, dpi=300, bbox_inches='tight')
plt.savefig(out_hm_pdf, bbox_inches='tight')
plt.show()
print(f"[OK] Heatmap salvate in:\n  - {out_hm_png}\n  - {out_hm_pdf}")

### 6.2 Generazione Grafici Comparativi Curve MoRF vs. LeRF (Task T6.2)
Visualizziamo le curve di degradazione progressiva a inferenza ($m=0..5$ campi neutralizzati a `0.5` su modelli congelati).

In [ ]:
# Degradation curves measured in Notebook 05
df_curves = pd.read_csv(METRICS_DIR / 'morf_lerf_curves.csv')
steps = [0, 1, 2, 3, 4, 5]
step_cols = [f'm={i}' for i in steps]
morf_data = {r['Model']: [r[c] for c in step_cols] for _, r in df_curves[df_curves['Curve'] == 'MoRF_F1'].iterrows()}
lerf_data = {r['Model']: [r[c] for c in step_cols] for _, r in df_curves[df_curves['Curve'] == 'LeRF_F1'].iterrows()}

plt.figure(figsize=(10, 6))
colors = {'Lopez17CNN': '#d95f02', 'BiflowTransformer': '#7570b3', 'MedDefenderTransformer': '#1b9e77'}

for net in morf_data.keys():
    plt.plot(steps, morf_data[net], marker='o', linewidth=2.2, color=colors[net], label=f"{net} (MoRF - Most Relevant)")
    plt.plot(steps, lerf_data[net], marker='s', linestyle='--', linewidth=1.8, color=colors[net], alpha=0.7, label=f"{net} (LeRF - Least Relevant)")

plt.axvline(x=1, color='gray', linestyle=':', linewidth=1.5, label='Passo m=1 (campo top)')
plt.title('Validazione Causale a Inferenza: Curve MoRF vs. LeRF a Modelli Congelati (RQ3)', fontsize=12, fontweight='bold')
plt.xlabel(r'Numero di Campi di Protocollo Neutralizzati a Inferenza, $m \in [0..5]$', fontsize=11)
plt.ylabel('Macro F1-Score [%]', fontsize=11)
plt.xticks(steps, ['m=0\n(Intatto)', 'm=1\n(campo top)', 'm=2', 'm=3', 'm=4', 'm=5'])
plt.ylim(0, 105)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='lower left', fontsize=9)

out_curves_png = FIGURES_DIR / 'morf_lerf_curves.png'
out_curves_pdf = FIGURES_DIR / 'morf_lerf_curves.pdf'
out_morf_vs_lerf_png = FIGURES_DIR / 'morf_vs_lerf_curves.png'
out_morf_vs_lerf_pdf = FIGURES_DIR / 'morf_vs_lerf_curves.pdf'

plt.savefig(out_curves_png, dpi=300, bbox_inches='tight')
plt.savefig(out_curves_pdf, bbox_inches='tight')
plt.savefig(out_morf_vs_lerf_png, dpi=300, bbox_inches='tight')
plt.savefig(out_morf_vs_lerf_pdf, bbox_inches='tight')
plt.show()
print(f"[OK] Grafici MoRF vs LeRF salvati in:\n  - {out_curves_png}\n  - {out_morf_vs_lerf_png}")

### 6.3 Matrici di Attenzione Inter-Pacchetto By-Design ($10 \times 10$) (Task T6.1b)
Visualizziamo le matrici di attenzione inter-pacchetto ($10 \times 10$) estratte durante il forward pass a costo zero:
- **BiflowTransformer**: matrice congiunta di auto-attenzione (*Attention Rollout*) tra tutti i pacchetti del flusso;
- **MedDefenderTransformer**: pesi di auto-attenzione del pooling *MHAN* tra i 10 pacchetti.
Tutti gli assi e le colorbar specificano **rigorosamente le unità di misura esplicite**.

In [ ]:
# Flow-averaged 10x10 inter-packet matrices on explained flows (extracted in Notebook 03)
T = 10
mat_biflow = npz_data['attn_biflow_mat']
mat_biflow = mat_biflow / mat_biflow.sum(axis=1, keepdims=True)  # submatrix without CLS: rows renormalized to sum to 1
mat_meddef = npz_data['attn_meddef_mat']
v_max = max(mat_biflow.max(), mat_meddef.max())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cbar_ax2 = fig.add_axes([0.92, 0.15, 0.015, 0.7])

# BiflowTransformer Rollout
sns.heatmap(
    mat_biflow, ax=axes[0], cmap='Blues', vmin=0, vmax=v_max,
    xticklabels=[f'P{j+1}' for j in range(T)], yticklabels=[f'P{i+1}' for i in range(T)],
    cbar=False, square=True, linewidths=0.5, linecolor='white'
)
axes[0].set_title('BiflowTransformer: Matrice Attention Rollout (By-Design)', fontweight='bold', fontsize=11)
axes[0].set_xlabel(r'Pacchetto Chiave/Valore (Key), $j \in [1..10]$', fontsize=10)
axes[0].set_ylabel(r'Pacchetto Interrogazione (Query), $i \in [1..10]$', fontsize=10)

# MedDefender MHAN
sns.heatmap(
    mat_meddef, ax=axes[1], cmap='Greens', vmin=0, vmax=v_max,
    xticklabels=[f'P{j+1}' for j in range(T)], yticklabels=[f'P{i+1}' for i in range(T)],
    cbar=True, cbar_ax=cbar_ax2, square=True, linewidths=0.5, linecolor='white'
)
axes[1].set_title('MedDefenderTransformer: Pesi di Pooling MHAN (By-Design)', fontweight='bold', fontsize=11)
axes[1].set_xlabel(r'Pacchetto Chiave/Valore (Key), $j \in [1..10]$', fontsize=10)
axes[1].set_ylabel(r'Pacchetto Interrogazione (Query), $i \in [1..10]$', fontsize=10)
cbar_ax2.set_ylabel(r'Peso di Attenzione Inter-Pacchetto $\alpha_{i,j}$ [Quota adimensionale, $\Sigma=1.0$]', fontsize=10)

plt.suptitle(r'Matrici di Auto-Attenzione Inter-Pacchetto ($10 \times 10$) - Spiegabilità Intrinseca a Costo Zero', fontsize=13, y=1.02)
out_attn_png = FIGURES_DIR / 'attention_matrices_inter_packet.png'
out_attn_pdf = FIGURES_DIR / 'attention_matrices_inter_packet.pdf'
plt.savefig(out_attn_png, dpi=300, bbox_inches='tight')
plt.savefig(out_attn_pdf, bbox_inches='tight')
plt.show()
print(f'[SUCCESS] Matrici di attenzione salvate in:\n  - {out_attn_png}\n  - {out_attn_pdf}')

### 6.4 Distribuzione Statistica dell'Importanza per Campo Header ($F=6$) (Task T6.1c)
Visualizziamo la dispersione statistica dell'importanza assoluta normalizzata $|\tilde{\phi}_f|$ sui singoli flussi del test set per ciascun campo di intestazione (`PL`, `IAT`, `DIR`, `WIN`, `TTL`, `FLG`).
Assi e legenda con **unità di misura esplicite**.

In [ ]:
# Collect per-sample distributions
npz_path = METRICS_DIR / 'xai_explanations.npz'
npz_data = np.load(npz_path, allow_pickle=True)
f_lopez = np.abs(npz_data['shap_lopez_field'])
f_biflow = np.abs(npz_data['shap_biflow_field'])
f_meddef = np.abs(npz_data['shap_meddef_field'])

records_df = []
for i in range(len(f_lopez)):
    for j, fld in enumerate(FIELDS):
        records_df.append({'Field': fld, 'Model': 'Lopez17CNN', 'Attribution': f_lopez[i, j]})
        records_df.append({'Field': fld, 'Model': 'BiflowTransformer', 'Attribution': f_biflow[i, j]})
        records_df.append({'Field': fld, 'Model': 'MedDefenderTransformer', 'Attribution': f_meddef[i, j]})

df_box = pd.DataFrame(records_df)

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_box, x='Field', y='Attribution', hue='Model', palette={'Lopez17CNN': '#d95f02', 'BiflowTransformer': '#7570b3', 'MedDefenderTransformer': '#1b9e77'})
plt.title('Distribuzione dell\'Importanza per Campo Header sui Flussi di Test (Eq. 5 Nascita et al.)', fontweight='bold', fontsize=12)
plt.xlabel(r'Campi di Intestazione Header Protocollo, $f \in \{PL, IAT, DIR, WIN, TTL, FLG\}$', fontsize=11)
plt.ylabel(r'Attribuzione Assoluta Normalizzata per Flusso $|\tilde{\phi}_f|$ [Quota adimensionale (a.u.)]', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5, axis='y')
plt.legend(title='Architettura di Rete', loc='upper right')

out_box_png = FIGURES_DIR / 'field_attribution_distributions_boxplot.png'
out_box_pdf = FIGURES_DIR / 'field_attribution_distributions_boxplot.pdf'
plt.savefig(out_box_png, dpi=300, bbox_inches='tight')
plt.savefig(out_box_pdf, bbox_inches='tight')
plt.show()
print(f'[SUCCESS] Boxplot di distribuzione salvato in:\n  - {out_box_png}\n  - {out_box_pdf}')

### 6.3 Compilazione Automatica del Report Sperimentale Consuntivo (Task T6.3)
Leggiamo i file CSV generati nelle Fasi 2, 4 e 5 ed esportiamo il consuntivo Markdown definitivo in `results/final_experimental_report.md`.

In [ ]:
# Tables generated by Notebooks 02, 04, and 05
df_bench = pd.read_csv(METRICS_DIR / 'benchmark_metrics.csv')
df_audit = pd.read_csv(METRICS_DIR / 'xai_audit_table.csv')
df_mask = pd.read_csv(METRICS_DIR / 'masking_results.csv')

report_md = rf"""**Progetto**: Group 6 - Marotta League - XAI for IoMT (DIETI UNINA)  
**Riferimento Master**: `results/final_experimental_report.md`  
**Dataset**: CIC-IoMT-2024 (13 classi multiclasse, $T=10$ pacchetti, $F=6$ campi header, `seed=0`)  

---

## 1. Benchmark Prestazionale ed Efficienza Edge CPU Single-Thread (Tabelle 1 e 2)
{df_bench.to_markdown(index=False)}

---

## 2. Audit Formale XAI: Concordanza By-Design vs. Post-Hoc (RQ1 e RQ2 - Tabella 3)
{df_audit.to_markdown(index=False)}

---

## 3. Validazione Causale a Modelli Congelati (RQ3 - Tabella 4)
{df_mask.to_markdown(index=False)}

"""

out_report_file = RESULTS_DIR / 'final_experimental_report.md'
with open(out_report_file, 'w', encoding='utf-8') as f:
    f.write(report_md)

print(f"\n[SUCCESS] Report consuntivo generato con successo in:\n  {out_report_file}")